# Notebook 4 — Multi-Temporal Landsat Data Preparation for LULC Analysis

## Objective

This notebook prepares multi-temporal Landsat imagery for Land Use/Land Cover (LULC) analysis within the established Gomti River–Lucknow study area.

Three Landsat observation years are used:

- **2003** — Landsat 7 ETM+
- **2014** — Landsat 8 OLI
- **2025** — Landsat 8/9 OLI

The imagery is used to investigate changes in land-cover conditions across the study period and to subsequently derive LULC-related variables for the flood-susceptibility analysis.

---

## Why These Three Years?

The study uses approximately decadal temporal intervals to examine long-term land-cover change:

**2003 → 2014 → 2025**

The 2003 imagery is from Landsat 7 ETM+ and the later observations are from the Landsat 8/9 OLI generation. Although the sensors differ, the imagery will be processed using a consistent workflow and the appropriate spectral bands for each sensor.

The selected scenes use the same WRS Path/Row:

**Path 144 / Row 041**

This provides a consistent spatial framework for the three temporal observations.

---

## Data Source

The Landsat imagery is obtained from the **USGS Landsat Collection 2 Level-2** products.

Level-2 products provide atmospherically corrected surface reflectance data and, where available, surface-temperature products. For this notebook, the surface-reflectance bands and quality-assurance layers are the primary inputs for LULC-related analysis.

---

## Dataset Organization

The raw Landsat datasets are stored separately by observation year:

```text
data/
└── raw/
    ├── 2003/
    │   └── Landsat 7 ETM+ scene
    │
    ├── 2014/
    │   └── Landsat 8 OLI scene
    │
    └── 2025/
        └── Landsat 8/9 OLI scene

## 1. Landsat Band Identification

The downloaded Landsat Collection 2 Level-2 products contain individual
GeoTIFF files for the spectral bands and quality-assurance layers.

Only the bands required for the LULC analysis are selected.

Because Landsat 7 ETM+ and Landsat 8/9 OLI use different band numbering,
the physical spectral role of each band is used when constructing the
multi-temporal dataset.

### Landsat 7 ETM+ — 2003

- Blue → B1
- Green → B2
- Red → B3
- NIR → B4
- SWIR 1 → B5
- SWIR 2 → B7

### Landsat 8/9 OLI — 2014 and 2025

- Blue → B2
- Green → B3
- Red → B4
- NIR → B5
- SWIR 1 → B6
- SWIR 2 → B7

The QA layers are retained for quality-control and cloud/shadow masking.

In [6]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

RAW_LANDSAT_DIR = PROJECT_ROOT / "data" / "raw"

years = [2003, 2014, 2025]

for year in years:

    year_dir = RAW_LANDSAT_DIR / str(year)

    print("\n" + "=" * 70)
    print(f"{year} LANDSAT FILES")
    print("=" * 70)

    for file in sorted(year_dir.rglob("*")):
        if file.is_file():
            print(file.name)


2003 LANDSAT FILES
LE07_L2SP_144041_20030221_20200916_02_T1_ANG.txt
LE07_L2SP_144041_20030221_20200916_02_T1_MTL.json
LE07_L2SP_144041_20030221_20200916_02_T1_MTL.txt
LE07_L2SP_144041_20030221_20200916_02_T1_MTL.xml
LE07_L2SP_144041_20030221_20200916_02_T1_QA_PIXEL.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_QA_RADSAT.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_SR_ATMOS_OPACITY.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_SR_B1.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_SR_B2.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_SR_B3.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_SR_B4.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_SR_B5.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_SR_B7.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_SR_CLOUD_QA.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_SR_stac.json
LE07_L2SP_144041_20030221_20200916_02_T1_ST_ATRAN.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_ST_B6.TIF
LE07_L2SP_144041_20030221_20200916_02_T1_ST_CDIST.TIF
LE07_L2SP_144041_20030221_20200916_0

## 2. Surface Reflectance Scaling and Quality Masking

The downloaded Landsat Collection 2 Level-2 surface-reflectance bands contain
scaled digital values rather than direct surface-reflectance values.

The Landsat Collection 2 scale factor and offset are therefore applied before
using the spectral bands for analysis.

Quality-assurance (QA) information is also used to identify pixels affected
by conditions such as:

- Cloud
- Cloud shadow
- Cirrus
- Snow/ice
- Other invalid observations

These pixels are excluded from the analysis.

The masking procedure is sensor-independent at the conceptual level, but the
actual Landsat QA bit structure is applied according to the corresponding
Collection 2 product.

The result is a quality-controlled surface-reflectance dataset suitable for
spectral-index calculation and subsequent LULC analysis.

In [7]:
from pathlib import Path
import rasterio
import numpy as np

# Raw Landsat directories
landsat_dirs = {
    2003: RAW_LANDSAT_DIR / "2003",
    2014: RAW_LANDSAT_DIR / "2014",
    2025: RAW_LANDSAT_DIR / "2025"
}

# Sensor-specific spectral bands
landsat_bands = {
    2003: {
        "blue": "SR_B1",
        "green": "SR_B2",
        "red": "SR_B3",
        "nir": "SR_B4",
        "swir1": "SR_B5",
        "swir2": "SR_B7"
    },
    2014: {
        "blue": "SR_B2",
        "green": "SR_B3",
        "red": "SR_B4",
        "nir": "SR_B5",
        "swir1": "SR_B6",
        "swir2": "SR_B7"
    },
    2025: {
        "blue": "SR_B2",
        "green": "SR_B3",
        "red": "SR_B4",
        "nir": "SR_B5",
        "swir1": "SR_B6",
        "swir2": "SR_B7"
    }
}

for year, bands in landsat_bands.items():

    print(f"\n{year}")
    print("-" * 40)

    for name, band in bands.items():
        print(f"{name:8s} → {band}")


2003
----------------------------------------
blue     → SR_B1
green    → SR_B2
red      → SR_B3
nir      → SR_B4
swir1    → SR_B5
swir2    → SR_B7

2014
----------------------------------------
blue     → SR_B2
green    → SR_B3
red      → SR_B4
nir      → SR_B5
swir1    → SR_B6
swir2    → SR_B7

2025
----------------------------------------
blue     → SR_B2
green    → SR_B3
red      → SR_B4
nir      → SR_B5
swir1    → SR_B6
swir2    → SR_B7


In [8]:
def find_landsat_file(year, band_name):
    """
    Find a Landsat surface-reflectance band inside the
    corresponding year directory.
    """

    year_dir = landsat_dirs[year]

    matches = list(
        year_dir.rglob(f"*_{band_name}.TIF")
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No {band_name} file found for {year}"
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple {band_name} files found for {year}:\n"
            + "\n".join(str(x) for x in matches)
        )

    return matches[0]


# Find all required spectral bands
band_files = {}

for year, bands in landsat_bands.items():

    band_files[year] = {}

    for name, band in bands.items():

        band_files[year][name] = find_landsat_file(
            year,
            band
        )

print("Required Landsat bands located successfully.")

Required Landsat bands located successfully.


In [9]:
for year in band_files:

    print(f"\n{year}")

    for name, path in band_files[year].items():
        print(f"{name:8s}: {path.name}")


2003
blue    : LE07_L2SP_144041_20030221_20200916_02_T1_SR_B1.TIF
green   : LE07_L2SP_144041_20030221_20200916_02_T1_SR_B2.TIF
red     : LE07_L2SP_144041_20030221_20200916_02_T1_SR_B3.TIF
nir     : LE07_L2SP_144041_20030221_20200916_02_T1_SR_B4.TIF
swir1   : LE07_L2SP_144041_20030221_20200916_02_T1_SR_B5.TIF
swir2   : LE07_L2SP_144041_20030221_20200916_02_T1_SR_B7.TIF

2014
blue    : LC08_L2SP_144041_20140211_20200911_02_T1_SR_B2.TIF
green   : LC08_L2SP_144041_20140211_20200911_02_T1_SR_B3.TIF
red     : LC08_L2SP_144041_20140211_20200911_02_T1_SR_B4.TIF
nir     : LC08_L2SP_144041_20140211_20200911_02_T1_SR_B5.TIF
swir1   : LC08_L2SP_144041_20140211_20200911_02_T1_SR_B6.TIF
swir2   : LC08_L2SP_144041_20140211_20200911_02_T1_SR_B7.TIF

2025
blue    : LC08_L2SP_144041_20250225_20250304_02_T1_SR_B2.TIF
green   : LC08_L2SP_144041_20250225_20250304_02_T1_SR_B3.TIF
red     : LC08_L2SP_144041_20250225_20250304_02_T1_SR_B4.TIF
nir     : LC08_L2SP_144041_20250225_20250304_02_T1_SR_B5.TIF
swir1 

In [10]:
def find_qa_pixel(year):

    year_dir = landsat_dirs[year]

    matches = list(
        year_dir.rglob("*_QA_PIXEL.TIF")
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"QA_PIXEL file not found for {year}"
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple QA_PIXEL files found for {year}"
        )

    return matches[0]


qa_pixel_files = {
    year: find_qa_pixel(year)
    for year in years
}

for year, path in qa_pixel_files.items():

    print(
        f"{year}: {path.name}"
    )

2003: LE07_L2SP_144041_20030221_20200916_02_T1_QA_PIXEL.TIF
2014: LC08_L2SP_144041_20140211_20200911_02_T1_QA_PIXEL.TIF
2025: LC08_L2SP_144041_20250225_20250304_02_T1_QA_PIXEL.TIF


In [11]:
def create_qa_mask(qa_array):
    """
    Create a valid-pixel mask from Landsat Collection 2 QA_PIXEL.

    True  = usable pixel
    False = pixel excluded from analysis
    """

    # QA_PIXEL bit positions
    fill = (qa_array & (1 << 0)) != 0
    dilated_cloud = (qa_array & (1 << 1)) != 0
    cirrus = (qa_array & (1 << 2)) != 0
    cloud = (qa_array & (1 << 3)) != 0
    cloud_shadow = (qa_array & (1 << 4)) != 0
    snow = (qa_array & (1 << 5)) != 0

    invalid = (
        fill
        | dilated_cloud
        | cirrus
        | cloud
        | cloud_shadow
        | snow
    )

    valid = ~invalid

    return valid

In [12]:
for year in years:

    with rasterio.open(qa_pixel_files[year]) as src:

        qa = src.read(1)

        valid_mask = create_qa_mask(qa)

        total_pixels = valid_mask.size
        valid_pixels = np.count_nonzero(valid_mask)
        invalid_pixels = total_pixels - valid_pixels

        valid_percent = (
            valid_pixels / total_pixels
        ) * 100

        invalid_percent = (
            invalid_pixels / total_pixels
        ) * 100

    print(f"\n{year}")
    print("-" * 40)
    print(f"Total pixels:   {total_pixels:,}")
    print(f"Valid pixels:   {valid_pixels:,}")
    print(f"Invalid pixels: {invalid_pixels:,}")
    print(f"Valid:          {valid_percent:.2f}%")
    print(f"Invalid:        {invalid_percent:.2f}%")


2003
----------------------------------------
Total pixels:   55,899,571
Valid pixels:   37,877,562
Invalid pixels: 18,022,009
Valid:          67.76%
Invalid:        32.24%

2014
----------------------------------------
Total pixels:   59,685,451
Valid pixels:   38,158,052
Invalid pixels: 21,527,399
Valid:          63.93%
Invalid:        36.07%

2025
----------------------------------------
Total pixels:   59,685,451
Valid pixels:   39,703,723
Invalid pixels: 19,981,728
Valid:          66.52%
Invalid:        33.48%


### QA Mask Diagnostic

The initial QA mask excludes pixels flagged by the Landsat Collection 2
QA_PIXEL layer.

The proportion of excluded pixels is relatively large despite the selected
scenes having low reported cloud cover. Therefore, the individual QA flags
are examined before finalizing the masking rule.

This diagnostic is performed to distinguish actual atmospheric contamination
from other QA conditions that may be present outside the study area.

The final mask will be selected based on the QA flags relevant to the LULC
analysis.

In [13]:
def qa_flag_counts(qa_array):

    flags = {
        "Fill":        (qa_array & (1 << 0)) != 0,
        "Dilated cloud": (qa_array & (1 << 1)) != 0,
        "Cirrus":      (qa_array & (1 << 2)) != 0,
        "Cloud":       (qa_array & (1 << 3)) != 0,
        "Cloud shadow": (qa_array & (1 << 4)) != 0,
        "Snow/Ice":    (qa_array & (1 << 5)) != 0
    }

    return {
        name: np.count_nonzero(mask)
        for name, mask in flags.items()
    }


for year in years:

    with rasterio.open(qa_pixel_files[year]) as src:
        qa = src.read(1)

    counts = qa_flag_counts(qa)

    print("\n" + "=" * 50)
    print(year)
    print("=" * 50)

    for flag, count in counts.items():
        print(f"{flag:16s}: {count:,}")


2003
Fill            : 17,985,701
Dilated cloud   : 17,094
Cirrus          : 0
Cloud           : 10,184
Cloud shadow    : 16,178
Snow/Ice        : 0

2014
Fill            : 19,170,146
Dilated cloud   : 420,663
Cirrus          : 633,039
Cloud           : 1,161,808
Cloud shadow    : 977,394
Snow/Ice        : 58

2025
Fill            : 19,154,076
Dilated cloud   : 279,709
Cirrus          : 61,905
Cloud           : 438,633
Cloud shadow    : 199,279
Snow/Ice        : 0


### QA Coverage Within the Study Area

The initial QA diagnostic was calculated for the complete Landsat scenes.
Because the analysis is restricted to the established study area, the
proportion of valid and invalid pixels is recalculated only within that
area.

This provides the relevant measure of image usability for the subsequent
LULC analysis.

In [14]:
import rasterio.mask

In [15]:
# Use the existing study-area geometry.
# Reproject it to each Landsat raster CRS before masking.

for year in years:

    with rasterio.open(qa_pixel_files[year]) as src:

        study_area_raster_crs = study_area.to_crs(src.crs)

        geometries = [
            geom.__geo_interface__
            for geom in study_area_raster_crs.geometry
        ]

        qa_clipped, clipped_transform = rasterio.mask.mask(
            src,
            geometries,
            crop=True,
            filled=False
        )

        qa = qa_clipped[0]

        valid_data = ~qa.mask

        qa_values = qa.data

        valid_mask = create_qa_mask(qa_values)

        # Only consider pixels that are actually inside
        # the study area.
        valid_mask &= valid_data

        study_pixels = np.count_nonzero(valid_data)
        usable_pixels = np.count_nonzero(valid_mask)
        unusable_pixels = study_pixels - usable_pixels

        usable_percent = (
            usable_pixels / study_pixels
        ) * 100

        unusable_percent = (
            unusable_pixels / study_pixels
        ) * 100

    print("\n" + "=" * 55)
    print(f"{year} — QA COVERAGE WITHIN STUDY AREA")
    print("=" * 55)

    print(f"Study-area pixels: {study_pixels:,}")
    print(f"Usable pixels:     {usable_pixels:,}")
    print(f"Unusable pixels:   {unusable_pixels:,}")
    print(f"Usable:            {usable_percent:.2f}%")
    print(f"Unusable:          {unusable_percent:.2f}%")


2003 — QA COVERAGE WITHIN STUDY AREA
Study-area pixels: 3,314,354
Usable pixels:     3,306,529
Unusable pixels:   7,825
Usable:            99.76%
Unusable:          0.24%

2014 — QA COVERAGE WITHIN STUDY AREA
Study-area pixels: 3,314,354
Usable pixels:     3,205,084
Unusable pixels:   109,270
Usable:            96.70%
Unusable:          3.30%

2025 — QA COVERAGE WITHIN STUDY AREA
Study-area pixels: 3,314,354
Usable pixels:     2,909,196
Unusable pixels:   405,158
Usable:            87.78%
Unusable:          12.22%


### QA Flag Diagnostic Within the Study Area

The 2025 scene contains a higher proportion of QA-excluded pixels within
the study area than the 2003 and 2014 observations.

The individual QA flags are therefore examined specifically within the
study area to determine whether the excluded pixels are primarily caused
by cloud-related contamination or by other invalid-data conditions.

This diagnostic will determine whether the current QA mask can be retained
for the 2025 observation or whether the scene requires further treatment.

In [16]:
def qa_flag_counts_within_study_area(year):

    with rasterio.open(qa_pixel_files[year]) as src:

        study_area_raster = study_area.to_crs(src.crs)

        geometries = [
            geom.__geo_interface__
            for geom in study_area_raster.geometry
        ]

        qa_clipped, _ = rasterio.mask.mask(
            src,
            geometries,
            crop=True,
            filled=False
        )

        qa = qa_clipped[0]

        inside = ~qa.mask
        qa_values = qa.data

        flags = {
            "Fill": (qa_values & (1 << 0)) != 0,
            "Dilated cloud": (qa_values & (1 << 1)) != 0,
            "Cirrus": (qa_values & (1 << 2)) != 0,
            "Cloud": (qa_values & (1 << 3)) != 0,
            "Cloud shadow": (qa_values & (1 << 4)) != 0,
            "Snow/Ice": (qa_values & (1 << 5)) != 0
        }

        counts = {}

        for name, flag in flags.items():
            counts[name] = np.count_nonzero(
                flag & inside
            )

        total_inside = np.count_nonzero(inside)

    return counts, total_inside


for year in years:

    counts, total_inside = (
        qa_flag_counts_within_study_area(year)
    )

    print("\n" + "=" * 60)
    print(f"{year} — QA FLAGS WITHIN STUDY AREA")
    print("=" * 60)

    for name, count in counts.items():

        percentage = (
            count / total_inside
        ) * 100

        print(
            f"{name:16s}: "
            f"{count:,} "
            f"({percentage:.2f}%)"
        )


2003 — QA FLAGS WITHIN STUDY AREA
Fill            : 6,505 (0.20%)
Dilated cloud   : 494 (0.01%)
Cirrus          : 0 (0.00%)
Cloud           : 230 (0.01%)
Cloud shadow    : 620 (0.02%)
Snow/Ice        : 0 (0.00%)

2014 — QA FLAGS WITHIN STUDY AREA
Fill            : 0 (0.00%)
Dilated cloud   : 21,635 (0.65%)
Cirrus          : 32,498 (0.98%)
Cloud           : 45,087 (1.36%)
Cloud shadow    : 48,910 (1.48%)
Snow/Ice        : 0 (0.00%)

2025 — QA FLAGS WITHIN STUDY AREA
Fill            : 0 (0.00%)
Dilated cloud   : 97,143 (2.93%)
Cirrus          : 59,721 (1.80%)
Cloud           : 285,899 (8.63%)
Cloud shadow    : 40,979 (1.24%)
Snow/Ice        : 0 (0.00%)


## 2. QA Masking — Result

The Landsat Collection 2 QA_PIXEL layer was used to exclude pixels affected
by invalid observations, clouds, cloud shadows, cirrus, and related quality
flags.

After restricting the assessment to the established study area:

| Year | Usable pixels | Unusable pixels |
|------|---------------:|----------------:|
| 2003 | 99.76% | 0.24% |
| 2014 | 96.70% | 3.30% |
| 2025 | 87.78% | 12.22% |

The 2003 and 2014 observations provide high usable coverage.

The 2025 observation contains substantially more cloud-related masking,
with cloud flags accounting for approximately 8.63% of the study-area
pixels, in addition to cirrus and cloud-shadow flags.

The QA mask is therefore retained without relaxing the quality criteria.
Cloud-contaminated pixels will remain excluded rather than being treated
as valid land-cover observations.

The 2025 scene will be retained for subsequent processing, with its
remaining valid pixels used for the LULC analysis.

## 3. Surface Reflectance Scaling

Landsat Collection 2 Level-2 surface-reflectance bands are stored as
scaled integer values.

The Collection 2 surface-reflectance scale factor and additive offset are
applied to convert the stored values into surface-reflectance values:

\[
SR = DN \times 0.0000275 - 0.2
\]

The transformation is applied consistently to the selected spectral bands
for 2003, 2014, and 2025.

Pixels excluded by the QA mask are assigned NoData values and are not used
in subsequent spectral-index calculations or LULC classification.

In [36]:
# ============================================================
# Step 3A — Scale and QA-mask Landsat surface reflectance
# ============================================================

scaled_bands = {}

SCALE_FACTOR = 0.0000275
ADD_OFFSET = -0.2

for year in years:

    scaled_bands[year] = {}

    # --------------------------------------------------------
    # Read the QA layer
    # --------------------------------------------------------
    with rasterio.open(qa_pixel_files[year]) as qa_src:

        qa = qa_src.read(1)

    qa_mask = create_qa_mask(qa)

    # --------------------------------------------------------
    # Process each required spectral band
    # --------------------------------------------------------
    for band_name, band_path in band_files[year].items():

        with rasterio.open(band_path) as src:

            dn = src.read(1).astype("float32")

        # Convert stored DN to surface reflectance
        reflectance = (
            dn * SCALE_FACTOR
            + ADD_OFFSET
        )

        # Remove invalid QA pixels
        reflectance[~qa_mask] = np.nan

        # Also remove physically invalid reflectance values
        reflectance[
           (reflectance < 0) | (reflectance > 1)
        ] = np.nan

        scaled_bands[year][band_name] = reflectance
    print(
        f"{year}: spectral bands scaled and QA-masked."
    )

2003: spectral bands scaled and QA-masked.
2014: spectral bands scaled and QA-masked.
2025: spectral bands scaled and QA-masked.


In [37]:
print(type(scaled_bands))
print(scaled_bands.keys())

for year in scaled_bands:

    print(f"\n{year}")

    for band_name, array in scaled_bands[year].items():

        valid = np.isfinite(array)

        print(
            f"{band_name:8s} | "
            f"min={np.nanmin(array):.4f} | "
            f"max={np.nanmax(array):.4f} | "
            f"valid={np.count_nonzero(valid):,}"
        )

<class 'dict'>
dict_keys([2003, 2014, 2025])

2003
blue     | min=0.0002 | max=0.3499 | valid=37,877,296
green    | min=0.0144 | max=0.4293 | valid=37,877,562
red      | min=0.0058 | max=0.4802 | valid=37,877,562
nir      | min=0.0193 | max=0.5882 | valid=37,877,562
swir1    | min=0.0006 | max=0.9407 | valid=37,877,556
swir2    | min=0.0012 | max=0.9928 | valid=37,877,365

2014
blue     | min=0.0000 | max=0.4333 | valid=38,157,865
green    | min=0.0009 | max=0.5619 | valid=38,158,044
red      | min=0.0001 | max=0.6668 | valid=38,158,033
nir      | min=0.0001 | max=0.8224 | valid=38,158,044
swir1    | min=0.0000 | max=0.9921 | valid=38,157,965
swir2    | min=0.0000 | max=0.9914 | valid=38,157,757

2025
blue     | min=0.0005 | max=0.4789 | valid=39,703,717
green    | min=0.0189 | max=0.4140 | valid=39,703,723
red      | min=0.0061 | max=0.4613 | valid=39,703,723
nir      | min=0.0158 | max=0.6672 | valid=39,703,723
swir1    | min=0.0005 | max=0.9977 | valid=39,703,708
swir2    | min=0.00

## 4. Spectral Index Calculation

Spectral indices transform combinations of Landsat spectral bands into
quantitative indicators that help distinguish different land-cover
characteristics.

Three indices are calculated for each observation year:

### NDVI — Normalized Difference Vegetation Index

NDVI is used to identify and characterize vegetation.

\[
NDVI = \frac{NIR - Red}{NIR + Red}
\]

Higher NDVI values generally indicate stronger vegetation response.

### NDBI — Normalized Difference Built-up Index

NDBI is used to enhance built-up and impervious surfaces.

\[
NDBI = \frac{SWIR1 - NIR}{SWIR1 + NIR}
\]

Higher values generally indicate stronger built-up/impervious surface
response.

### MNDWI — Modified Normalized Difference Water Index

MNDWI is used to enhance open-water features.

\[
MNDWI = \frac{Green - SWIR1}{Green + SWIR1}
\]

Higher values generally indicate stronger water response.

The indices are calculated using the physically corresponding spectral
bands for each Landsat sensor rather than using identical band numbers
across sensors.

All calculations use the QA-masked surface-reflectance data produced in
Step 3.

In [38]:
# ============================================================
# Step 4 — Calculate NDVI, NDBI and MNDWI
# ============================================================

indices = {}

for year in years:

    # Get the scaled surface-reflectance bands
    bands = scaled_bands[year]

    blue = bands["blue"]
    green = bands["green"]
    red = bands["red"]
    nir = bands["nir"]
    swir1 = bands["swir1"]

    # --------------------------------------------------------
    # NDVI
    # --------------------------------------------------------
    ndvi_denominator = nir + red

    ndvi = np.full(
        nir.shape,
        np.nan,
        dtype="float32"
    )

    valid_ndvi = (
        np.isfinite(nir)
        & np.isfinite(red)
        & (ndvi_denominator != 0)
    )

    ndvi[valid_ndvi] = (
        (nir[valid_ndvi] - red[valid_ndvi])
        / ndvi_denominator[valid_ndvi]
    )

    # --------------------------------------------------------
    # NDBI
    # --------------------------------------------------------
    ndbi_denominator = swir1 + nir

    ndbi = np.full(
        nir.shape,
        np.nan,
        dtype="float32"
    )

    valid_ndbi = (
        np.isfinite(swir1)
        & np.isfinite(nir)
        & (ndbi_denominator != 0)
    )

    ndbi[valid_ndbi] = (
        (swir1[valid_ndbi] - nir[valid_ndbi])
        / ndbi_denominator[valid_ndbi]
    )

    # --------------------------------------------------------
    # MNDWI
    # --------------------------------------------------------
    mndwi_denominator = green + swir1

    mndwi = np.full(
        green.shape,
        np.nan,
        dtype="float32"
    )

    valid_mndwi = (
        np.isfinite(green)
        & np.isfinite(swir1)
        & (mndwi_denominator != 0)
    )

    mndwi[valid_mndwi] = (
        (green[valid_mndwi] - swir1[valid_mndwi])
        / mndwi_denominator[valid_mndwi]
    )

    # --------------------------------------------------------
    # IMPORTANT: actually store the results
    # --------------------------------------------------------
    indices[year] = {
        "NDVI": ndvi,
        "NDBI": ndbi,
        "MNDWI": mndwi
    }

    print(
        f"{year}: NDVI, NDBI and MNDWI calculated."
    )

2003: NDVI, NDBI and MNDWI calculated.
2014: NDVI, NDBI and MNDWI calculated.
2025: NDVI, NDBI and MNDWI calculated.


In [39]:
def normalized_difference(band_a, band_b):
    """
    Calculate a normalized difference index while
    avoiding division by zero.
    """
    
    denominator = band_a + band_b

    result = np.full(
        band_a.shape,
        np.nan,
        dtype="float32"
    )

    valid = (
        np.isfinite(band_a)
        & np.isfinite(band_b)
        & (denominator != 0)
    )

    result[valid] = (
        (band_a[valid] - band_b[valid])
        / denominator[valid]
    )

    return result

In [40]:
for year in years:

    print("\n" + "=" * 50)
    print(year)
    print("=" * 50)

    year_indices = indices.get(year)

    if year_indices is None:
        print(f"No calculated indices found for {year}.")
        continue

    for index_name, array in year_indices.items():

        valid = np.isfinite(array)

        if np.count_nonzero(valid) == 0:
            print(f"{index_name}: no valid pixels")
            continue

        print(
            f"{index_name}: "
            f"min={np.nanmin(array):.3f}, "
            f"max={np.nanmax(array):.3f}, "
            f"mean={np.nanmean(array):.3f}, "
            f"valid={np.count_nonzero(valid):,}"
        )


2003
NDVI: min=-0.542, max=0.942, mean=0.593, valid=37,877,562
NDBI: min=-0.991, max=0.536, mean=-0.227, valid=37,877,556
MNDWI: min=-0.796, max=0.990, mean=-0.410, valid=37,877,556

2014
NDVI: min=-0.996, max=0.998, mean=0.606, valid=38,158,025
NDBI: min=-1.000, max=0.953, mean=-0.231, valid=38,157,957
MNDWI: min=-0.991, max=1.000, mean=-0.385, valid=38,157,957

2025
NDVI: min=-0.507, max=0.944, mean=0.599, valid=39,703,723
NDBI: min=-0.980, max=0.738, mean=-0.258, valid=39,703,708
MNDWI: min=-0.884, max=0.979, mean=-0.376, valid=39,703,708


## 5. Save Spectral Indices

The calculated NDVI, NDBI, and MNDWI rasters are saved as GeoTIFF
products for each observation year.

The outputs retain the spatial reference, raster dimensions, and
georeferencing of the corresponding Landsat surface-reflectance data.

These index rasters will be used as input features for the subsequent
LULC classification and temporal land-cover analysis.

Output structure:

data/
└── processed/
    └── notebook_04/
        ├── 2003/
        ├── 2014/
        └── 2025/

In [41]:
from pathlib import Path

NOTEBOOK_04_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

for year in years:
    (NOTEBOOK_04_DIR / str(year)).mkdir(
        parents=True,
        exist_ok=True
    )

print("Notebook 4 output directories ready.")

Notebook 4 output directories ready.


In [42]:
import rasterio

for year in years:

    # Use one original spectral band as the spatial reference
    reference_band = band_files[year]["red"]

    with rasterio.open(reference_band) as src:

        profile = src.profile.copy()

    # Store floating-point index values
    profile.update(
        dtype="float32",
        count=1,
        nodata=np.nan,
        compress="lzw"
    )

    for index_name, array in indices[year].items():

        output_path = (
            NOTEBOOK_04_DIR
            / str(year)
            / f"{index_name}_{year}.tif"
        )

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(
                array.astype("float32"),
                1
            )

        print(
            f"Saved: {output_path}"
        )

Saved: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\2003\NDVI_2003.tif
Saved: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\2003\NDBI_2003.tif
Saved: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\2003\MNDWI_2003.tif
Saved: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\2014\NDVI_2014.tif
Saved: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\2014\NDBI_2014.tif
Saved: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\2014\MNDWI_2014.tif
Saved: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\2025\NDVI_2025.tif
Saved: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\2025\NDBI_2025.tif
Saved: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\2025\MNDWI_2025.tif


In [43]:
for year in years:

    print(f"\n{year}")

    year_dir = NOTEBOOK_04_DIR / str(year)

    for file in sorted(year_dir.glob("*.tif")):
        print(file.name)


2003
MNDWI_2003.tif
NDBI_2003.tif
NDVI_2003.tif

2014
MNDWI_2014.tif
NDBI_2014.tif
NDVI_2014.tif

2025
MNDWI_2025.tif
NDBI_2025.tif
NDVI_2025.tif


## 6. LULC Feature Stack Preparation

The LULC classification requires multiple spectral and spectral-index
features rather than a single raster layer.

For each observation year, the following nine features are prepared:

### Spectral bands

- Blue
- Green
- Red
- NIR
- SWIR1
- SWIR2

### Spectral indices

- NDVI
- NDBI
- MNDWI

The spectral bands provide the original multispectral information, while
the indices provide enhanced information related to vegetation, built-up
surfaces, and water.

The same feature structure is maintained across 2003, 2014, and 2025.
Sensor-specific band numbering has already been handled during the
previous processing steps.

These features will form the input variables for the subsequent LULC
classification.

In [44]:
# ============================================================
# Step 6A — Create LULC feature stacks
# ============================================================

feature_stacks = {}

feature_names = [
    "blue",
    "green",
    "red",
    "nir",
    "swir1",
    "swir2",
    "NDVI",
    "NDBI",
    "MNDWI"
]

for year in years:

    feature_stacks[year] = {}

    # Spectral bands
    for band_name in [
        "blue",
        "green",
        "red",
        "nir",
        "swir1",
        "swir2"
    ]:
        feature_stacks[year][band_name] = (
            scaled_bands[year][band_name]
        )

    # Spectral indices
    feature_stacks[year]["NDVI"] = (
        indices[year]["NDVI"]
    )

    feature_stacks[year]["NDBI"] = (
        indices[year]["NDBI"]
    )

    feature_stacks[year]["MNDWI"] = (
        indices[year]["MNDWI"]
    )

    print(
        f"{year}: {len(feature_stacks[year])} features prepared."
    )

2003: 9 features prepared.
2014: 9 features prepared.
2025: 9 features prepared.


In [45]:
# ============================================================
# Step 6B — Verify feature dimensions
# ============================================================

for year in years:

    print("\n" + "=" * 50)
    print(year)
    print("=" * 50)

    shapes = {}

    for name, array in feature_stacks[year].items():

        shapes[name] = array.shape

        print(
            f"{name:8s}: {array.shape}"
        )

    unique_shapes = set(shapes.values())

    if len(unique_shapes) == 1:
        print("All features have identical dimensions.")
    else:
        print("WARNING: Feature dimensions do not match.")


2003
blue    : (7111, 7861)
green   : (7111, 7861)
red     : (7111, 7861)
nir     : (7111, 7861)
swir1   : (7111, 7861)
swir2   : (7111, 7861)
NDVI    : (7111, 7861)
NDBI    : (7111, 7861)
MNDWI   : (7111, 7861)
All features have identical dimensions.

2014
blue    : (7801, 7651)
green   : (7801, 7651)
red     : (7801, 7651)
nir     : (7801, 7651)
swir1   : (7801, 7651)
swir2   : (7801, 7651)
NDVI    : (7801, 7651)
NDBI    : (7801, 7651)
MNDWI   : (7801, 7651)
All features have identical dimensions.

2025
blue    : (7801, 7651)
green   : (7801, 7651)
red     : (7801, 7651)
nir     : (7801, 7651)
swir1   : (7801, 7651)
swir2   : (7801, 7651)
NDVI    : (7801, 7651)
NDBI    : (7801, 7651)
MNDWI   : (7801, 7651)
All features have identical dimensions.


## 7. LULC Classification Scheme

A supervised land-use/land-cover (LULC) classification will be developed
for 2003, 2014, and 2025.

Five LULC classes are defined:

| Class ID | Class | Description |
|----------|-------|-------------|
| 1 | Water | Rivers, ponds, lakes and other open-water surfaces |
| 2 | Built-up | Buildings, roads and other predominantly impervious urban surfaces |
| 3 | Vegetation | Forest, dense vegetation, parks and other non-agricultural vegetated areas |
| 4 | Agriculture | Cultivated and agricultural land |
| 5 | Bare/Open land | Exposed soil, barren surfaces and other sparsely vegetated open land |

The same class definitions are maintained across all three observation
years to support temporal comparison.

The nine spectral features prepared in Step 6 will be used as predictor
variables for supervised classification.

Training samples must represent the spectral characteristics of each class
and must be spatially distributed across the study area.

In [46]:
# ============================================================
# Step 7 — LULC Classification Scheme
# ============================================================

# LULC class IDs
LULC_CLASSES = {
    1: "Water",
    2: "Built-up",
    3: "Vegetation",
    4: "Agriculture",
    5: "Bare/Open land"
}

# Reverse lookup: class name → class ID
LULC_CLASS_IDS = {
    name: class_id
    for class_id, name in LULC_CLASSES.items()
}

print("LULC classification scheme:")
print("-" * 40)

for class_id, class_name in LULC_CLASSES.items():
    print(f"{class_id}: {class_name}")

LULC classification scheme:
----------------------------------------
1: Water
2: Built-up
3: Vegetation
4: Agriculture
5: Bare/Open land


## 8. Training and Reference Sample Creation

Supervised LULC classification requires reference samples with known land-cover
classes.

Training samples are created separately for 2003, 2014, and 2025 while
maintaining the same five LULC class definitions established in Step 7.

The five classes are:

1. Water
2. Built-up
3. Vegetation
4. Agriculture
5. Bare/Open land

Reference samples should be spatially distributed throughout the study area
and should represent spectrally homogeneous examples of their respective
classes.

The reference samples are used to train the supervised classification model.
They must be independent from the samples later used for accuracy assessment
to avoid overly optimistic classification accuracy.

Training samples are based on visual/reference interpretation of the
corresponding year's imagery rather than labels generated from the same
spectral features being classified.

In [47]:
# ============================================================
# Step 8 — Training / Reference Sample Configuration
# ============================================================

# LULC classes defined in Step 7
training_classes = {
    1: "Water",
    2: "Built-up",
    3: "Vegetation",
    4: "Agriculture",
    5: "Bare/Open land"
}

# Observation years
training_years = [2003, 2014, 2025]

# Minimum target number of reference polygons per class.
# These are targets for creating a reasonably distributed
# training dataset, not automatically generated samples.
MIN_POLYGONS_PER_CLASS = 10

print("Training sample configuration")
print("-" * 50)

print("Years:", training_years)

print("\nClasses:")
for class_id, class_name in training_classes.items():
    print(f"{class_id}: {class_name}")

print(
    f"\nTarget minimum polygons per class: "
    f"{MIN_POLYGONS_PER_CLASS}"
)

Training sample configuration
--------------------------------------------------
Years: [2003, 2014, 2025]

Classes:
1: Water
2: Built-up
3: Vegetation
4: Agriculture
5: Bare/Open land

Target minimum polygons per class: 10


In [48]:
# ============================================================
# Define locations for reference polygons
# ============================================================

TRAINING_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
    / "training_samples"
)

TRAINING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

training_files = {
    year: TRAINING_DIR / f"LULC_training_{year}.gpkg"
    for year in training_years
}

print("\nExpected training polygon files:")

for year, path in training_files.items():
    print(f"{year}: {path}")


Expected training polygon files:
2003: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\training_samples\LULC_training_2003.gpkg
2014: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\training_samples\LULC_training_2014.gpkg
2025: d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\training_samples\LULC_training_2025.gpkg


## 9. Training Sample Extraction

The manually interpreted reference polygons prepared for each observation
year are used to extract pixel-level training samples from the LULC feature
stack.

For every valid pixel contained within a reference polygon, the following
nine predictor variables are extracted:

- Blue
- Green
- Red
- NIR
- SWIR1
- SWIR2
- NDVI
- NDBI
- MNDWI

The LULC class assigned to the reference polygon is attached to every
extracted pixel.

Pixels containing NoData or invalid values are excluded.

The resulting table contains one row per valid training pixel and will be
used as the input dataset for the supervised LULC classifier.

Training samples are maintained separately for 2003, 2014, and 2025 so that
the classifier for each observation year is trained using reference data
corresponding to that year's land-cover conditions.

In [50]:
import geopandas as gpd
import pandas as pd
from rasterio.features import geometry_mask

In [52]:
training_polygons = {}

for year in training_years:

    path = training_files[year]

    if not path.exists():
        print(
            f"WARNING: Training polygon file not found for {year}:\n"
            f"{path}"
        )
        continue

    gdf = gpd.read_file(path)

    training_polygons[year] = gdf

    print(
        f"{year}: "
        f"{len(gdf)} training polygons loaded."
    )

if not training_polygons:
    print(
        "No training polygon files were found. "
        "Create the GeoPackage files in the training_samples folder "
        "before continuing with sample extraction."
    )

d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\training_samples\LULC_training_2003.gpkg
d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\training_samples\LULC_training_2014.gpkg
d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_04\training_samples\LULC_training_2025.gpkg
No training polygon files were found. Create the GeoPackage files in the training_samples folder before continuing with sample extraction.


In [53]:
for year, gdf in training_polygons.items():

    print("\n" + "=" * 50)
    print(year)
    print("=" * 50)

    print(
        gdf["class_id"]
        .value_counts()
        .sort_index()
    )